# SRA 多领域法学硕士教程

在本笔记本中，您将在小规模的LLM中体验SRA“同时学习多个领域（代码、数学、文本）”的特色。
使用存储库中“data/lang/”中的三种类型的数据作为数据集，我们将可视化 SRA 如何自动划分（专门化）每个域的突触。

In [ ]:
# Colab環境でのみ実行してください（ローカル環境の場合はスキップ可）
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch tiktoken matplotlib seaborn

# パスの追加
sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

## 1. 加载和标记数据集
加载`data/lang/`目录下的`code.txt`、`math.txt`、`text.txt`。

In [ ]:
import torch
import tiktoken
import sys

tokenizer = tiktoken.get_encoding("cl100k_base")
vocab_size = tokenizer.n_vocab

domains = ["code", "math", "text"]
data = {}

base_dir = "." if 'google.colab' in sys.modules else ".."

for d in domains:
    path = f"{base_dir}/data/lang/{d}.txt"
    with open(path, "r", encoding="utf-8") as f:
        # 少しデータをカサ増しして学習しやすくする
        text = (f.read() + "\n") * 5
    tokens = tokenizer.encode(text, allowed_special="all")
    data[d] = torch.tensor(tokens, dtype=torch.long)
    print(f"Domain '{d}': {len(tokens)} tokens")

## 2. 构建模型
和以前一样，创建一个具有较小配置的“MoESRALanguageModel”。

In [ ]:
from sra_language_models import MoESRALanguageModel

dim = 128
layers = 2
num_synapses = 16
k = 2
syn_hidden = 256
max_seq_len = 64

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"使用デバイス: {device}")

model = MoESRALanguageModel(
    vocab_size=vocab_size,
    dim=dim,
    layers=layers,
    num_synapses=num_synapses,
    k=k,
    syn_hidden=syn_hidden,
    max_seq_len=max_seq_len
).to(device)

## 3. 多领域学习循环
在每一步中，我们随机选择一个域并从该文本中创建小批量来进行学习。

In [ ]:
import random
import torch.nn.functional as F
from sra_experiment import make_optimizer, load_balance_loss

def get_multidomain_batch(data_dict, batch_size, seq_len):
    x = torch.zeros((batch_size, seq_len), dtype=torch.long)
    y = torch.zeros((batch_size, seq_len), dtype=torch.long)
    batch_domains = []
    
    for i in range(batch_size):
        d = random.choice(list(data_dict.keys()))
        batch_domains.append(d)
        d_data = data_dict[d]
        max_start = len(d_data) - seq_len - 1
        start = random.randint(0, max(0, max_start))
        
        x[i] = d_data[start:start+seq_len]
        y[i] = d_data[start+1:start+seq_len+1]
        
    return x.to(device), y.to(device), batch_domains

# 学習パラメータ
batch_size = 32
steps = 400
lr = 5e-4
opt = make_optimizer(model, lr)

model.train()
for step in range(1, steps + 1):
    x, y, b_domains = get_multidomain_batch(data, batch_size, max_seq_len)
    
    logits, router_logits = model(x, dense=False)
    
    ce_loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    lb_loss = load_balance_loss(router_logits)
    loss = ce_loss + 0.1 * lb_loss
    
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    
    if step % 50 == 0:
        print(f"Step {step}/{steps} | Loss: {loss.item():.4f}")

## 4. 按域的路由可视化
将每个域的样本文本输入到经过训练的模型中，然后聚合并可视化总体上常用的突触。您可以查看SRA的“专业化”。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sra_experiment import usage_stats

model.eval()
domain_usages = {}

with torch.no_grad():
    for d in domains:
        # 評価用に数バッチ回して使用状況を集計
        total_usage = None
        for _ in range(5):
            x, y, _ = get_multidomain_batch({d: data[d]}, batch_size, max_seq_len)
            _, router_logits = model(x)
            
            u = usage_stats(router_logits) # shape: (num_synapses,)
            total_usage = u if total_usage is None else total_usage + u
            
        # 正規化して保存
        domain_usages[d] = (total_usage / total_usage.sum()).cpu().numpy()

# ヒートマップ描画 (ドメイン x シナプス)
usage_matrix = np.array([domain_usages[d] for d in domains])

plt.figure(figsize=(10, 4))
sns.heatmap(usage_matrix, cmap="Blues", annot=True, fmt=".2f", yticklabels=domains)
plt.title("Synapse Usage per Domain")
plt.xlabel("Synapse ID")
plt.ylabel("Domain")
plt.show()